## **1. Đọc dữ liệu các mã cổ phiếu từ yahoo finance**



In [ ]:
import yfinance as yf
import pandas as pd
import matplotlib.pyplot as plt
from google.colab import files
import os

# Tạo thư mục lưu hình nếu chưa có
os.makedirs("Stock_Plots", exist_ok=True)

# Danh sách mã cổ phiếu
symbols = ['AAPL', 'MSFT', 'GOOG', 'META', 'AMZN', 'NVDA', 'TSLA', 'AMD', 'INTC', 'ADBE',
           'JPM', 'BAC', 'GS', 'MS', 'V', 'MA', 'PYPL',
           'WMT', 'TGT', 'COST', 'NKE', 'SBUX', 'MCD', 'KO', 'PEP',
           'GM', 'F', 'RIVN', 'LCID',
           'EL', 'ELF', 'ULTA', 'COTY', 'PG', 'LRLCY',
           'JNJ', 'PFE', 'MRNA', 'UNH', 'ABBV', 'LLY',
           'XOM', 'CVX', 'BP', 'COP', 'NEE',
           'NFLX', 'DIS', 'WBD', 'SONY', 'CMCSA',
           'HD', 'LOW', 'LEN', 'DHI', 'PLD',
           'CRM', 'NOW', 'ORCL', 'SHOP', 'SNOW',
           'VFS', 'BABA', 'TSM', 'NVS', 'SHEL']

start_date = '2015-01-01'
end_date = '2025-10-01'

file_path = "Stocks_MultiSheet_(1_1_2015 to 1_10_2025).xlsx"
summary_data = []

with pd.ExcelWriter(file_path, engine="openpyxl") as writer:
    for sym in symbols:
        df = yf.download(sym, start=start_date, end=end_date, progress=False, auto_adjust=False)

        if not df.empty:
            # Giữ lại cột Adj Close (giá đã điều chỉnh)
            df = df[['Open', 'High', 'Low', 'Close', 'Adj Close', 'Volume']]

            # Ghi dữ liệu vào sheet riêng
            df.to_excel(writer, sheet_name=sym)

            # Tạo summary cho sheet này
            increase_pct = (df['Adj Close'].iloc[-1] - df['Adj Close'].iloc[0]) / df['Adj Close'].iloc[0] * 100
            summary = {
                'Symbol': sym,
                'Adj Close Mean': df['Adj Close'].mean(),
                'Adj Close Max': df['Adj Close'].max(),
                'Adj Close Min': df['Adj Close'].min(),
                'Adj Close Std': df['Adj Close'].std(),
                'Total Volume': df['Volume'].sum(),
                'Increase (%)': increase_pct
            }
            summary_data.append(summary)

            # Vẽ biểu đồ Close và Adj Close
            plt.figure(figsize=(10,5))
            plt.plot(df.index, df['Close'], label='Close', alpha=0.7)
            plt.plot(df.index, df['Adj Close'], label='Adj Close', linestyle='--', alpha=0.8)
            plt.title(f'{sym} Close vs Adj Close ({start_date} - {end_date})')
            plt.xlabel('Date')
            plt.ylabel('Price (USD)')
            plt.grid(True)
            plt.legend()
            plt.tight_layout()
            plt.show()  # Hiển thị trong notebook
            plt.savefig(f'Stock_Plots/{sym}_Close_AdjClose.png')  # Lưu hình
            plt.close()

    # Ghi summary vào sheet tổng hợp
    if summary_data:
        summary_df = pd.DataFrame(summary_data)
        summary_df.to_excel(writer, sheet_name="Summary", index=False)

# Tải file Excel về máy
files.download(file_path)

##  **2. Design trading strategies using technical indicators**

### **2.1. Tạo 1 data frame từ các mã cổ phiếu**

In [ ]:
# Tạo file csv
# Import file dữ liệu excel đã tạo, chuyển thành file csv
!pip install openpyxl
import pandas as pd
from google.colab import files
import io
from scipy.stats.mstats import winsorize
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression

In [ ]:
uploaded = files.upload()
file_name = next(iter(uploaded)) # Lấy tên file đã upload

Saving Stocks_MultiSheet_(1_1_2015 to 1_10_2025).xlsx to Stocks_MultiSheet_(1_1_2015 to 1_10_2025) (2).xlsx


In [ ]:
# Đọc tất cả các sheets. Kết quả là một dictionary {tên_sheet: DataFrame}
# Nếu dùng files.upload() (byte data), hãy dùng io.BytesIO(uploaded[file_name])
# Nếu dùng Google Drive (đường dẫn), hãy dùng file_name
try:
    if 'uploaded' in locals() or 'uploaded' in globals(): # Kiểm tra xem đã upload file chưa
        xls_dict = pd.read_excel(io.BytesIO(uploaded[file_name]), sheet_name=None)
    else:
        xls_dict = pd.read_excel(file_name, sheet_name=None)
except Exception as e:
    print(f"Lỗi khi đọc file Excel: {e}")
    # Xử lý trường hợp không tìm thấy file nếu dùng Google Drive
    # hoặc file_name không đúng nếu dùng files.upload()
    exit()

# Lặp qua dictionary và lưu từng DataFrame thành file CSV
for sheet_name, df in xls_dict.items():
    # Tên file CSV sẽ là tên sheet (mã cổ phiếu)
    csv_file_name = f'{sheet_name}.csv'

    # Sử dụng .to_csv để xuất DataFrame thành file CSV
    # index=False để không bao gồm chỉ mục (index) của DataFrame vào file CSV
    df.to_csv(csv_file_name, index=False, encoding='utf-8')

In [ ]:
import pandas as pd
import glob

# Đọc tất cả file CSV trong thư mục hiện tại
csv_files = glob.glob("*.csv")
dataframes = {}

for file in csv_files:
    # Đọc tất cả file CSV trong thư mục hiện tại
    ticker = file.split(".")[0]

    # Đọc file, bỏ 3 hàng đầu (Price, Ticker, header trống)
    df = pd.read_csv(file, skiprows=3, header=None)

    # Tự đặt lại header chuẩn
    df.columns = ['Date', 'Open', 'High', 'Low', 'Close', 'Adj Close', 'Volume']

    # Lưu lại vào dict
    dataframes[ticker] = df

# ✅ Kiểm tra
print(dataframes["GOOG"].tail())

                     Date        Open        High         Low       Close  \
2697  2025-09-24 00:00:00  252.149994  252.960007  247.220001  247.830002   
2698  2025-09-25 00:00:00  244.839996  247.315002  241.649994  246.570007   
2699  2025-09-26 00:00:00  247.785004  250.119995  246.639999  247.179993   
2700  2025-09-29 00:00:00  248.274994  251.647995  243.199997  244.360001   
2701  2025-09-30 00:00:00  243.059998  243.675003  239.570999  243.550003   

       Adj Close    Volume  
2697  247.830002  16958500  
2698  246.570007  17379800  
2699  247.179993  16594600  
2700  244.360001  23157200  
2701  243.550003  22541200  


### **2.2. Simple Moving Averages:** (Technical Indicators)
- Khi đường trung bình ngắn hạn (short-term MA) vượt lên đường trung bình dài hạn (long-term MA) $⇒$ xu hướng tăng $⇒$ MUA.
- Khi MA ngắn hạn (short-term MA) cắt xuống MA dài hạn (long-term MA) $⇒$ xu hướng giảm $⇒$ BÁN.

**VD:** MA ngắn (ví dụ 20 ngày) phản ứng nhanh với giá hiện tại.

MA dài (ví dụ 50 ngày) phản ứng chậm hơn.

Khi MA ngắn vượt MA dài, chứng tỏ giá đang tăng nhanh hơn xu hướng trung bình => tín hiệu mua (golden cross).

Khi ngược lại, ta có bán (death cross)

In [ ]:
# Tạo 1 data frame copy
df_copy = dataframes.copy()
# In thử 1 mã cổ phiếu để kiểm tra
df_copy["AAPL"]

,Date,Open,High,Low,Close,Adj Close,Volume
0,2015-01-02,27.847500,27.860001,26.837500,27.332500,24.261049,212818400
1,2015-01-05,27.072500,27.162500,26.352501,26.562500,23.577578,257142000
2,2015-01-06,26.635000,26.857500,26.157499,26.565001,23.579794,263188400
3,2015-01-07,26.799999,27.049999,26.674999,26.937500,23.910429,160423600
4,2015-01-08,27.307501,28.037500,27.174999,27.972500,24.829130,237458000
...,...,...,...,...,...,...,...
2697,2025-09-24,255.220001,255.740005,251.039993,252.309998,252.309998,42303700
2698,2025-09-25,253.210007,257.170013,251.710007,256.869995,256.869995,55202100
2699,2025-09-26,254.100006,257.600006,253.779999,255.460007,255.460007,46076300
2700,2025-09-29,254.559998,255.000000,253.009995,254.429993,254.429993,40127700


In [ ]:
# Tính SMA của các mã cổ phiếu
for ticker, df in df_copy.items():

    # 1.1. Chuẩn hóa Index và Kiểu Dữ liệu
    if 'Date' in df.columns:

        df['Date'] = pd.to_datetime(df['Date'], format='%Y-%m-%d %H:%M:%S', errors='coerce')

        # Loại bỏ các hàng mà cột 'Date' là NaT (chứa mã cổ phiếu hoặc dữ liệu rác)
        df.dropna(subset=['Date'], inplace=True)

        # Đặt cột 'Date' đã được làm sạch làm Index
        df = df.set_index('Date')

    # 1.2. Chuyển các cột giá trị sang kiểu số
    df['Adj Close'] = pd.to_numeric(df['Adj Close'], errors='coerce')
    df['Volume'] = pd.to_numeric(df['Volume'], errors='coerce')

    # Loại bỏ các hàng có NaN nếu Adj Close không hợp lệ
    df.dropna(subset=['Adj Close'], inplace=True)

    # 1.3. Tính toán SMA
    df['SMA_20'] = df['Adj Close'].rolling(window=20).mean()
    df['SMA_50'] = df['Adj Close'].rolling(window=50).mean()

    # 1.4 Tạo Tín hiệu (Signal)
    # Logic: Giữ lệnh MUA (1) khi SMA ngắn > SMA dài, ngược lại là 0
    df['Signal'] = np.where(df['SMA_20'] > df['SMA_50'], 1, 0)

    # 1.5. Cập nhật lại DataFrame đã tính toán vào dictionary
    df_copy[ticker] = df

In [ ]:
# TÍNH ĐỘ CHÍNH XÁC (BACKTEST) CHO MỖI MÃ CỔ PHIẾU NẾU DÙNG SIGNAL TRÊN
accuracy_results = []

for ticker, df in df_copy.items():
    if ticker == 'Summary':
        continue

    # 1. Chuẩn bị dữ liệu để so sánh (Shift data)
    # Lấy giá đóng cửa ngày hôm sau (Next Day Price)
    df['Next_Close'] = df['Adj Close'].shift(-1)

    # Xác định xu hướng THỰC TẾ của ngày hôm sau (1: Tăng, 0: Giảm/Đi ngang)
    df['Actual_Trend'] = np.where(df['Next_Close'] > df['Adj Close'], 1, 0)

    # 2. So sánh Tín hiệu dự báo với Thực tế
    # Đúng khi: (Signal=1 VÀ Giá Tăng) HOẶC (Signal=0 VÀ Giá Giảm)
    df['Is_Correct'] = df['Signal'] == df['Actual_Trend']

    # 3. Loại bỏ dữ liệu nhiễu
    # Cần bỏ 49 dòng đầu (do chưa có SMA_50) và dòng cuối cùng (do chưa có Next_Close)
    df_clean = df.dropna(subset=['SMA_50', 'Next_Close'])

    # 4. Tính Win Rate
    if len(df_clean) > 0:
        win_rate = df_clean['Is_Correct'].mean() * 100
    else:
        win_rate = 0

    # Lưu kết quả
    accuracy_results.append({
        'Symbol': ticker,
        'Win Rate (%)': round(win_rate, 2),
        'Total_Days': len(df_clean)
    })

    # Cập nhật lại df (để có thể soi chi tiết từng mã nếu muốn)
    df_copy[ticker] = df

# --- HIỂN THỊ KẾT QUẢ ---
df_accuracy = pd.DataFrame(accuracy_results)
df_accuracy = df_accuracy.sort_values(by='Win Rate (%)', ascending=False)

print(f"📊 Bảng xếp hạng độ chính xác (Win Rate) trên {len(df_accuracy)} mã cổ phiếu:")
print(df_accuracy.head(10)) # In top 10 mã chuẩn nhất
print(df_accuracy.tail(5)) # In 5 mã tệ nhất

📊 Bảng xếp hạng độ chính xác (Win Rate) trên 66 mã cổ phiếu:
   Symbol  Win Rate (%)  Total_Days
45      V         52.83        2652
46    MCD         52.22        2652
43     MA         52.19        2652
61   COST         51.92        2652
17    TSM         51.89        2652
50    JPM         51.77        2652
41   SNOW         51.73        1216
44    NEE         51.73        2652
60    WMT         51.70        2652
62   PYPL         51.54        2526
   Symbol  Win Rate (%)  Total_Days
55   LCID         49.09        1214
25   NFLX         48.68        2652
16    AMD         48.64        2652
57    WBD         48.45        2652
9     VFS         46.65         971


In [ ]:
# In thử 1 mã cổ phiếu để kiểm tra
df_copy["AAPL"][-8:-2]

,Open,High,Low,Close,Adj Close,Volume,SMA_20,SMA_50,Signal,Next_Close,Actual_Trend,Is_Correct
Date,,,,,,,,,,,,
2025-09-19,241.229996,246.300003,240.210007,245.500000,245.500000,163741300,234.371001,223.828286,1,256.079987,1,True
2025-09-22,248.300003,256.640015,248.119995,256.079987,256.079987,105517400,235.787000,224.731473,1,254.429993,0,False
2025-09-23,255.880005,257.339996,253.580002,254.429993,254.429993,60275200,237.150500,225.652403,1,252.309998,0,False
2025-09-24,255.220001,255.740005,251.039993,252.309998,252.309998,42303700,238.300500,226.521144,1,256.869995,1,True
2025-09-25,253.210007,257.170013,251.710007,256.869995,256.869995,55202100,239.619499,227.460109,1,255.460007,0,False
2025-09-26,254.100006,257.600006,253.779999,255.460007,255.460007,46076300,240.764500,228.373670,1,254.429993,0,False


### **2.3. Upper/Lower Bollinger Band** (Technical Indicators)
- Upper Band = SMA + 2$\sigma$

- Lower Band = SMA -2$\sigma$
- Khi giá cao quá Upper Band, giá cao bất thường.
Khi giá thấp quá Lower band, giá thấp bất thường.

Dựa vào indicator này, có thể xây dựng được các tín hiệu **Mean Reversion** hoặc **Breakout** tùy ngữ cảnh.

**1.1 Mean Reversion**

- Buy (Mua) khi giá đóng cửa < Lower Band → kỳ vọng bật lên.

- Sell (Bán) khi giá đóng cửa > Upper Band → kỳ vọng điều chỉnh xuống.

- Thoát lệnh khi giá quay lại vùng MA (hoặc MA ± 0.5σ).

**1.2 Breakout**

- Buy khi giá đóng cửa > Upper Band → breakout lên.

- Sell khi giá đóng cửa < Lower Band → breakout xuống.

- Thoát lệnh khi giá quay lại trong dải.

In [ ]:
# Tính Bollinger Band của các mã cổ phiếu
for ticker, df in df_copy.items():
    if ticker == 'Summary': continue

    # 1.1. Tính toán độ lệch chuẩn
    df['STD_20'] = df['Adj Close'].rolling(window=20).std()
    df['STD_50'] = df['Adj Close'].rolling(window=50).std()

    # 1.2. Tính Upper Band và Lower Band
    df['UpperBand_20'] = df['SMA_20'] + 2 * df['STD_20']
    df['LowerBand_20'] = df['SMA_20'] - 2 * df['STD_20']

    df['UpperBand_50'] = df['SMA_50'] + 2 * df['STD_50']
    df['LowerBand_50'] = df['SMA_50'] - 2 * df['STD_50']

    # 1.3. Cập nhật lại DataFrame đã tính toán vào dictionary
    df_copy[ticker] = df

**Xây dựng Win Rate cho chiến lược Mean Reversion**

In [ ]:
cols_to_drop = [
    'Signal', 'Actual_Trend', 'Is_Correct', 'Next_Close', 'Is_Correct'
]

In [ ]:
import pandas as pd
import numpy as np

df_copy2 = {}

for ticker, df in df_copy.items():
    # Tạo bản sao độc lập
    df_new = df.copy()
    df_new = df_new.drop(columns=cols_to_drop, errors='ignore')
    df_copy2[ticker] = df_new

for ticker, df in df_copy2.items():

    if ticker == 'Summary': continue

    # Tạo Tín hiệu (Mean Reversion Strategy)
    conditions = [
        (df['Adj Close'] < df['LowerBand_20']), # Điều kiện 1: Giá rớt khỏi dây dưới -> MUA
        (df['Adj Close'] > df['UpperBand_20'])  # Điều kiện 2: Giá vượt dây trên -> BÁN
    ]

    # Kết quả tương ứng với điều kiện: 1 (Tăng/Mua), -1 (Giảm/Bán)
    choices = [1, -1]

    # Mặc định là 0 (Không làm gì khi giá nằm giữa)
    df['Signal'] = np.select(conditions, choices, default=0)

    df_copy2[ticker] = df

# TÍNH WIN RATE (CHỈ TÍNH NHỮNG NGÀY CÓ TÍN HIỆU)
accuracy_results = []

for ticker, df in df_copy2.items():
    if ticker == 'Summary': continue

    # 1. Lấy biến động thực tế ngày hôm sau
    df['Next_Close'] = df['Adj Close'].shift(-1)

    # Nếu giá mai > giá nay là 1 (Tăng), ngược lại là -1 (Giảm)
    df['Actual_Trend'] = np.where(df['Next_Close'] > df['Adj Close'], 1, -1)

    # Lọc ra những ngày CÓ RA TÍN HIỆU (Bỏ qua những ngày Signal = 0)
    df_trade = df[df['Signal'] != 0].copy()

    # Loại bỏ dòng cuối cùng bị NaN do shift
    df_trade = df_trade.dropna(subset=['Next_Close'])

    # So sánh
    df_trade['Is_Correct'] = df_trade['Signal'] == df_trade['Actual_Trend']

    # Tính Win Rate
    if len(df_trade) > 0:
        win_rate = df_trade['Is_Correct'].mean() * 100
    else:
        win_rate = 0 # Không có điểm vào lệnh nào

    accuracy_results.append({
        'Symbol': ticker,
        'Win Rate (%)': round(win_rate, 2),
        'Total_Trades': len(df_trade) # Số lần vào lệnh
    })

# Hiển thị kết quả
df_bb = pd.DataFrame(accuracy_results)
df_bb = df_bb.sort_values(by='Win Rate (%)', ascending=False)

print(f"📊 Hiệu quả chiến thuật Mean Reversion (Bollinger Bands):")
print(df_bb.head(10))

📊 Hiệu quả chiến thuật Mean Reversion (Bollinger Bands):
   Symbol  Win Rate (%)  Total_Trades
9     VFS         67.37            95
41   SNOW         62.30           122
28   META         59.36           283
36   RIVN         59.09            88
64    CRM         56.77           266
12     KO         56.74           282
62   PYPL         56.34           284
45      V         55.47           265
47    UNH         55.34           309
26    LEN         55.22           268


**Xây dựng Win Rate cho chiến lược Mean Reversion**

In [ ]:
cols_to_drop = [
    'Signal', 'Actual_Trend', 'Is_Correct', 'Next_Close', 'Is_Correct'
]

In [ ]:
import pandas as pd
import numpy as np

df_copy3 = {}

for ticker, df in df_copy.items():
    # Tạo bản sao độc lập
    df_new = df.copy()
    df_new = df_new.drop(columns=cols_to_drop, errors='ignore')

    df_copy3[ticker] = df_new

for ticker, df in df_copy3.items():

    if ticker == 'Summary': continue

    # Tạo Tín hiệu (Mean Reversion Strategy)
    conditions = [
        (df['Adj Close'] > df['UpperBand_20']), # Điều kiện 1: Giá > Upper -> MUA (1)
        (df['Adj Close'] < df['LowerBand_20'])  # Điều kiện 2: Giá < Lower -> BÁN (-1)
    ]

    # Kết quả tương ứng với điều kiện: 1 (Tăng/Mua), -1 (Giảm/Bán)
    choices = [1, -1]

    # Mặc định là 0 (Không làm gì khi giá nằm giữa)
    df['Signal'] = np.select(conditions, choices, default=0)

    df_copy3[ticker] = df

# TÍNH WIN RATE (CHỈ TÍNH NHỮNG NGÀY CÓ TÍN HIỆU)
accuracy_results = []

for ticker, df in df_copy3.items():
    if ticker == 'Summary': continue

    # 1. Lấy biến động thực tế ngày hôm sau
    df['Next_Close'] = df['Adj Close'].shift(-1)

    # Nếu giá mai > giá nay là 1 (Tăng), ngược lại là -1 (Giảm)
    df['Actual_Trend'] = np.where(df['Next_Close'] > df['Adj Close'], 1, -1)

    # Lọc ra những ngày CÓ RA TÍN HIỆU (Bỏ qua những ngày Signal = 0)
    df_trade = df[df['Signal'] != 0].copy()

    # Loại bỏ dòng cuối cùng bị NaN do shift
    df_trade = df_trade.dropna(subset=['Next_Close'])

    # So sánh
    df_trade['Is_Correct'] = df_trade['Signal'] == df_trade['Actual_Trend']

    # Tính Win Rate
    if len(df_trade) > 0:
        win_rate = df_trade['Is_Correct'].mean() * 100
    else:
        win_rate = 0 # Không có điểm vào lệnh nào

    accuracy_results.append({
        'Symbol': ticker,
        'Win Rate (%)': round(win_rate, 2),
        'Total_Trades': len(df_trade) # Số lần vào lệnh
    })

# Hiển thị kết quả
df_bb = pd.DataFrame(accuracy_results)
df_bb = df_bb.sort_values(by='Win Rate (%)', ascending=False)

print(f"📊 Hiệu quả chiến thuật BreakOut (Bollinger Bands):")
print(df_bb.head(10))

📊 Hiệu quả chiến thuật BreakOut (Bollinger Bands):
   Symbol  Win Rate (%)  Total_Trades
4    TSLA         57.67           326
61   COST         55.63           311
13   SHOP         55.41           296
10     BP         55.15           301
3    AAPL         53.16           316
14   ORCL         53.11           322
48   INTC         53.02           315
63     HD         52.96           287
6     JNJ         52.30           283
18    PLD         51.92           287


### **2.4. Log Return**
- Lợi suất Logarit có tính cộng gộp tuyệt đối. Tổng lợi suất Logarit của nhiều kỳ chính là logarit của giá cuối cùng chia cho giá ban đầu.

- Việc cộng dồn Log Returns rất thuận tiện cho việc tính toán hiệu suất tích lũy dài hạn và mô hình hóa chuỗi thời gian, vì có thể cộng chúng lại để tìm tổng lợi suất của một danh mục đầu tư. (để phù hợp dữ liệu Time Series ở Lab 05).
- Nhiều mô hình thống kê (bao gồm Hồi quy) hoạt động tốt nhất khi dữ liệu đầu vào tuân theo Phân phối Chuẩn (Normal Distribution). Mà do tính cộng gộp, $\log(p_{t})$ có xu hướng phân phối chuẩn (tức là $p_{t}$ có xu hướng phân phối Log-Normal). Làm cho các kiểm định thống kê và mô hình hóa trở nên đáng tin cậy hơn.
- Đảm bảo rằng các kiểm định xác thực mô hình hồi quy (ví dụ: Breusch-Pagan Test) trở nên đáng tin cậy hơn.

In [ ]:
# Tính log return của mỗi mã cổ phiếu
import numpy as np

for ticker, df in df_copy.items():

    # 1.1 Tính Log Returns (Lợi suất Logarit)
    df['Log_Returns'] = np.log(df['Adj Close'] / df['Adj Close'].shift(1))
    # 1.2 Cập nhật lại DataFrame đã tính toán vào dictionary
    df_copy[ticker] = df

In [ ]:
# In để kiểm tra
df_copy['AAPL']

,Open,High,Low,Close,Adj Close,Volume,SMA_20,SMA_50,STD_20,STD_50,UpperBand_20,LowerBand_20,UpperBand_50,LowerBand_50,Log_Returns
Date,,,,,,,,,,,,,,,
2015-01-02,27.847500,27.860001,26.837500,27.332500,24.261049,212818400,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2015-01-05,27.072500,27.162500,26.352501,26.562500,23.577578,257142000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.028576
2015-01-06,26.635000,26.857500,26.157499,26.565001,23.579794,263188400,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000094
2015-01-07,26.799999,27.049999,26.674999,26.937500,23.910429,160423600,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.013925
2015-01-08,27.307501,28.037500,27.174999,27.972500,24.829130,237458000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.037703
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-09-24,255.220001,255.740005,251.039993,252.309998,252.309998,42303700,238.300500,226.521144,8.181692,13.472069,254.663883,221.937116,253.465282,199.577006,-0.008367
2025-09-25,253.210007,257.170013,251.710007,256.869995,256.869995,55202100,239.619499,227.460109,8.946886,13.920156,257.513272,221.725727,255.300420,199.619797,0.017912
2025-09-26,254.100006,257.600006,253.779999,255.460007,255.460007,46076300,240.764500,228.373670,9.447231,14.231692,259.658961,221.870038,256.837055,199.910286,-0.005504


### **2.5. Volatility**
- Độ lệch chuẩn ($\sigma$) của Log Returns trong một cửa sổ cố định (ví dụ: 20 hoặc 50 ngày).
- Dùng để đo rủi ro và là thành phần quan trọng trong Bollinger Bands. (đã tính ở phần 2)

### **2.6. Turnover**
Doanh số phản ánh giá trị tiền tệ thực tế được giao dịch, giúp so sánh hoạt động giao dịch công bằng hơn giữa các công ty (quan trọng khi xử lý Thinly Traded Stocks - Cổ phiếu giao dịch thưa thớt) (Ở lab 04).

In [ ]:
# Tính doanh số Turnover = Volumn * Close
import numpy as np

for ticker, df in df_copy.items():

    # 1.1 Tính doanh số TurnOver
    df['Turnover'] = df['Volume']*df['Close']
    # 1.2 Cập nhật lại DataFrame đã tính toán vào dictionary
    df_copy[ticker] = df

In [ ]:
# In để kiểm tra
df_copy['AAPL']

,Open,High,Low,Close,Adj Close,Volume,SMA_20,SMA_50,STD_20,STD_50,UpperBand_20,LowerBand_20,UpperBand_50,LowerBand_50,Log_Returns,Turnover
Date,,,,,,,,,,,,,,,,
2015-01-02,27.847500,27.860001,26.837500,27.332500,24.261049,212818400,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.816859e+09
2015-01-05,27.072500,27.162500,26.352501,26.562500,23.577578,257142000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.028576,6.830334e+09
2015-01-06,26.635000,26.857500,26.157499,26.565001,23.579794,263188400,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000094,6.991600e+09
2015-01-07,26.799999,27.049999,26.674999,26.937500,23.910429,160423600,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.013925,4.321411e+09
2015-01-08,27.307501,28.037500,27.174999,27.972500,24.829130,237458000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.037703,6.642294e+09
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-09-24,255.220001,255.740005,251.039993,252.309998,252.309998,42303700,238.300500,226.521144,8.181692,13.472069,254.663883,221.937116,253.465282,199.577006,-0.008367,1.067365e+10
2025-09-25,253.210007,257.170013,251.710007,256.869995,256.869995,55202100,239.619499,227.460109,8.946886,13.920156,257.513272,221.725727,255.300420,199.619797,0.017912,1.417976e+10
2025-09-26,254.100006,257.600006,253.779999,255.460007,255.460007,46076300,240.764500,228.373670,9.447231,14.231692,259.658961,221.870038,256.837055,199.910286,-0.005504,1.177065e+10


## **3. Design momentum strategies using price-volume data**

### **3.1. Momentum tích lũy (Cumulative Momentum)**
- Thay vì chỉ xét một bước thay đổi của 1 khoảng thời gian ngắn, ta cộng dồn (tích lũy) momentum qua thời gian để đo độ mạnh liên tục của xu hướng.

=> Có thể coi như là tổng hiệu suất liên tục của giá trong một khoảng cửa sổ trượt (rolling window).

**Momentum theo hướng xu hướng** (trend-following): Giá có đà tăng mạnh → tiếp tục mua; đà giảm mạnh → tiếp tục bán.
- Khi CumMomentum > ngưỡng dương → xu hướng tăng → Tín hiệu buy.

- Khi CumMomentum < ngưỡng âm → xu hướng giảm → Tín hiệu sell.

**Momentum đảo chiều** (mean reversion):
Động lượng tích lũy quá lớn thì sắp đảo chiều.
- Khi CumMomentum quá cao → thị trường quá nóng → Bán.

- Khi CumMomentum quá thấp → thị trường quá nguội → Mua.


**Nguyên tắc cơ bản**

- Window (số ngày tích lũy) càng ngắn → tín hiệu nhạy hơn, nhưng nhiễu hơn.

- Window càng dài → tín hiệu ổn định hơn, nhưng chậm phản ứng.

Ta cần chọn sao cho vừa đủ nhạy để bắt xu hướng, không bị nhiễu do dao động nhỏ.
=> Chọn 120 ngày, vừa đủ mạnh mẽ ước lượng thị trường, vừa không quá nhiễu do ít ngày.

In [ ]:
# Tính Momentum tích lũy 120 ngày
for ticker, df in df_copy.items():

    # 1.1 Tính Momentum tích lũy trong 120 ngày từ Log_return đã tính
    df['Momentum_120D'] = df['Log_Returns'].rolling(window=120).sum()

    # 1.2 Gán lại DataFrame vào dictionary
    df_copy[ticker] = df

In [ ]:
# In để kiểm tra
df_copy['AAPL']

,Open,High,Low,Close,Adj Close,Volume,SMA_20,SMA_50,STD_20,STD_50,UpperBand_20,LowerBand_20,UpperBand_50,LowerBand_50,Log_Returns,Turnover,Momentum_120D
Date,,,,,,,,,,,,,,,,,
2015-01-02,27.847500,27.860001,26.837500,27.332500,24.261049,212818400,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.816859e+09,NaN
2015-01-05,27.072500,27.162500,26.352501,26.562500,23.577578,257142000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.028576,6.830334e+09,NaN
2015-01-06,26.635000,26.857500,26.157499,26.565001,23.579794,263188400,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000094,6.991600e+09,NaN
2015-01-07,26.799999,27.049999,26.674999,26.937500,23.910429,160423600,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.013925,4.321411e+09,NaN
2015-01-08,27.307501,28.037500,27.174999,27.972500,24.829130,237458000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.037703,6.642294e+09,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-09-24,255.220001,255.740005,251.039993,252.309998,252.309998,42303700,238.300500,226.521144,8.181692,13.472069,254.663883,221.937116,253.465282,199.577006,-0.008367,1.067365e+10,0.121948
2025-09-25,253.210007,257.170013,251.710007,256.869995,256.869995,55202100,239.619499,227.460109,8.946886,13.920156,257.513272,221.725727,255.300420,199.619797,0.017912,1.417976e+10,0.236873
2025-09-26,254.100006,257.600006,253.779999,255.460007,255.460007,46076300,240.764500,228.373670,9.447231,14.231692,259.658961,221.870038,256.837055,199.910286,-0.005504,1.177065e+10,0.307049
